# 1) Imports and Setup

In [1]:
import sys
import os
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import scipy.cluster.hierarchy as sch
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans
from adjustText import adjust_text

# Get the absolute path to the project root
project_root = os.path.dirname(os.path.dirname(os.getcwd()))
models_path = os.path.join(project_root, 'models')
sys.path.append(models_path)

from DKT.dkt_model import DKT

# 2) Model and Data Imports


In [2]:
# Load the JSON file
best_config = {'learning_rate': 0.01,
               'model_params':
               {
            "num_skills": 15,
            "num_other": 5,
            "embed_dim": 7,
            "hid_size": 150,
            "num_hid_layers": 1,
            "drop_prob": 0.5
            }
               }
skills_file_path = os.path.join(os.getcwd(), '..', '..', 'data', 'preprocessed', 'df_skill_names.csv')
df_skill_names = pd.read_csv(skills_file_path)

skills_to_keep = [
    'Box and Whisker', 'Circle Graph', 'Histogram as Table or Graph', 'Number Line', 'Scatter Plot', 
        'Stem and Leaf Plot', 'Table', 'Venn Diagram', 'Mean', 'Median', 'Mode', 'Range',
    'Probability of Two Distinct Events', 'Probability of a Single Event', 'D.4.8-understanding-concept-of-probabilities'
]

df_skill_names = df_skill_names[df_skill_names['skill_name'].isin(skills_to_keep)]

num_skills = best_config['model_params']['num_skills']
embedding_dim = best_config['model_params']['embed_dim']


model = DKT(**best_config['model_params'])
model.load_state_dict(torch.load('dkt_full_model_skills_15.pth', map_location=torch.device('cpu')))
# Set the model to evaluation model
model.eval()


C:\Users\Botond\AppData\Local\Temp\ipykernel_7068\3653837917.py:29: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load('dkt_full_model_skills_15.

DKT(
  (embedding): CustomEmbedding(
    (linear): Linear(in_features=15, out_features=7, bias=False)
  )
  (rnn): LSTM(12, 150, batch_first=True)
  (dropout): Dropout(p=0.5, inplace=False)
  (out): Linear(in_features=150, out_features=15, bias=True)
  (sigmoid): Sigmoid()
)

# 3) Displaying Relations

In [16]:
def get_relations(model, skills_to_keep):

    df = pd.DataFrame(.0, columns=skills_to_keep, index=skills_to_keep)

    for i in range(model.num_skills):
        skills = torch.zeros((1, 1, num_skills)).to(dtype=torch.float)
        skills[0, 0, i] = 1  # Set the specific skill index to 1

        other = torch.tensor([[1, 0.66, 0, 0, 0.14]]).unsqueeze(0).to(dtype=torch.float)
        lengths = torch.tensor([1]).to(dtype=torch.long)

        output = model(skills, other, lengths)

        df.iloc[i, :] = output.squeeze().detach().cpu().numpy()

    return df


In [ ]:
def highlight_cells(val, threshold):
    color = 'red' if val > threshold else 'white'
    return f'background-color: {color}'


In [24]:
df = get_relations(model, skills_to_keep)
threshold = 0.8

styled_df = df.style.map(lambda x: highlight_cells(x, threshold))
styled_df


,Box and Whisker,Circle Graph,Histogram as Table or Graph,Number Line,Scatter Plot,Stem and Leaf Plot,Table,Venn Diagram,Mean,Median,Mode,Range,Probability of Two Distinct Events,Probability of a Single Event,D.4.8-understanding-concept-of-probabilities
Box and Whisker,0.822329,0.482246,0.684416,0.546780,0.474090,0.885449,0.680082,0.642982,0.214733,0.878300,0.678962,0.160710,0.369044,0.673137,0.539518
Circle Graph,0.416767,0.168742,0.521346,0.391883,0.699937,0.314442,0.436844,0.414539,0.265975,0.581906,0.823358,0.448313,0.254036,0.572052,0.727790
Histogram as Table or Graph,0.604473,0.224953,0.588142,0.438106,0.650179,0.703946,0.598139,0.466261,0.097426,0.666331,0.880481,0.280840,0.136944,0.692736,0.744560
Number Line,0.373372,0.146800,0.509103,0.431099,0.581390,0.233261,0.351111,0.362734,0.282830,0.478314,0.763141,0.426487,0.267416,0.480193,0.663562
Scatter Plot,0.264912,0.110650,0.421369,0.360833,0.794348,0.042878,0.300466,0.480984,0.633049,0.505938,0.755898,0.589152,0.462625,0.386702,0.673448
Stem and Leaf Plot,0.368212,0.091778,0.533190,0.413458,0.722173,0.821931,0.570458,0.158400,0.014032,0.400873,0.945301,0.447885,0.030940,0.740912,0.835648
Table,0.187570,0.064500,0.481792,0.364071,0.708261,0.202665,0.294737,0.234505,0.097020,0.358203,0.896832,0.402915,0.091343,0.577673,0.811280
Venn Diagram,0.379706,0.140319,0.492223,0.382941,0.634894,0.195914,0.349647,0.437166,0.306476,0.479457,0.780294,0.391112,0.268329,0.500127,0.672933
Mean,0.337776,0.148179,0.453294,0.404894,0.649317,0.084829,0.298763,0.451069,0.612585,0.504087,0.640198,0.540458,0.525135,0.343193,0.544067
Median,0.292661,0.128137,0.512895,0.407227,0.518785,0.177722,0.261994,0.324726,0.268681,0.377697,0.752055,0.409034,0.225515,0.447420,0.664033
